# 🐐 Livestock Auction Market Tracker

## Project Overview
**Target Source:** Uvalde Market Report (Southwest Livestock Exchange)
**Objective:** Automate the extraction, parsing, and formatting of sheep and goat pricing data from weekly PDF market reports. This data pipeline feeds into an aggregated historical tracking system to support livestock pricing and sale decisions.

**Author:** [Your Name]
**Last Updated:** May 1, 2026

---

### 🗂️ Notebook Architecture

This pipeline is modularized into distinct, targeted operations to ensure stability and ease of troubleshooting:

* **Step 1: Environment Configuration** Installs required external libraries (`pdfplumber`) for deep text extraction.
* **Step 2: Kernel Refresh**
    Restarts the Python environment to formally initialize new library dependencies.
* **Step 3: PySpark Data Blueprint** Establishes the structured schema (Columns: Location, Date, Animal Class, Weight, Prices) required for the final DataFrame.
* **Step 4: Web Navigation & Extraction Engine** Executes standard HTTP requests to locate the most recent PDF report, downloads it into memory, and extracts the raw textual data.
* **Step 5: Data Parsing (Pending)** Regular Expressions (Regex) logic to isolate specific goat classes (e.g., Billies, Nannies, Cabritos) and their corresponding market values.

In [0]:
#Step 1: Environment Configuration Installs required external libraries (pdfplumber) for deep text extraction.
%pip install pdfplumber


In [0]:
#Step 2: Kernel Refresh Restarts the Python environment to formally initialize new library dependencies.
%restart_python

In [0]:
#Step 3: PySpark Data Blueprint Establishes the structured schema (Columns: Location, Date, Animal Class, Weight, Prices) required for the final DataFrame.
from pyspark.sql.types import StructType, StructField, StringType, FloatType, DateType

# 1. Define the blueprint for the auction data
auction_schema = StructType([
    StructField("Auction_Location", StringType(), True),
    StructField("Sale_Date", DateType(), True),
    StructField("Animal_Class", StringType(), True),
    StructField("Weight_Range_lbs", StringType(), True),
    StructField("Price_Low", FloatType(), True),
    StructField("Price_High", FloatType(), True),
    StructField("Price_Unit", StringType(), True)
])

# 2. Create the empty prototype table
prototype_df = spark.createDataFrame([], auction_schema)

# 3. Display the table so we can verify the columns
display(prototype_df)

In [0]:
#Step 4: Web Navigation & Extraction Engine Executes standard HTTP requests to locate the most recent PDF report, downloads it into memory, and extracts the raw textual data.
import requests
from bs4 import BeautifulSoup
import pdfplumber
from io import BytesIO

# 1. Target the Uvalde Market Report webpage (Corrected URL)
uvalde_url = "https://www.southwestlivestock.com/market-report/" 
response = requests.get(uvalde_url)
soup = BeautifulSoup(response.content, "html.parser")

# 2. Hunt for the most recent PDF link on the page
uvalde_pdf_link = None
for link in soup.find_all('a', href=True):
    if '.pdf' in link['href'].lower():
        uvalde_pdf_link = link['href']
        
        # Handle relative URLs just in case the site formats them weirdly
        if not uvalde_pdf_link.startswith('http'):
            uvalde_pdf_link = "https://www.southwestlivestock.com" + uvalde_pdf_link
        break

if uvalde_pdf_link:
    print(f"Target Acquired: {uvalde_pdf_link}")
    
    # 3. Download the Uvalde PDF directly into memory
    pdf_response = requests.get(uvalde_pdf_link)
    
    # 4. Open the PDF and extract the text
    with pdfplumber.open(BytesIO(pdf_response.content)) as pdf:
        for i, page in enumerate(pdf.pages):
            print(f"\n--- Scraping Uvalde Report: Page {i+1} ---")
            
            # Extract raw text to see how the goat prices are formatted
            text = page.extract_text()
            print(text)

else:
    print("Mission failed: No PDF market reports found on the Uvalde page.")

In [0]:
#Step 5: Data Parsing (Pending) Regular Expressions (Regex) logic to isolate specific goat classes (e.g., Billies, Nannies, Cabritos) and their corresponding market values.
import re
import datetime

# 1. Define the specific goat categories we want to track for the app
target_categories = [
    "Packer Nannies", "Fat Spanish Nannies", "Stocker Nannies",
    "Good Cabritos", "Small Cabritos", "Billies"
]

# 2. Set up an empty list to hold our cleanly parsed rows
parsed_data = []

# (For the prototype, we will hardcode the date from the report. 
# Later, we can automate this date extraction too!)
report_date = datetime.date(2026, 4, 21) 

# 3. Process the raw 'text' variable (which was saved into memory from Step 4)
for line in text.split('\n'):
    line = line.strip() # Clean up any trailing spaces
    
    # Check if the current line contains one of our target goat categories
    if any(cat in line for cat in target_categories):
        
        # The Regex Magic: 
        # This looks for the category name, skips the dots/spaces, 
        # and grabs the two numbers following the dollar signs.
        match = re.search(r"([A-Za-z\s]+?)[\.…\s]+\$(\d+)\s*to\s*\$(\d+)", line)
        
        if match:
            animal_class = match.group(1).strip()
            price_low = float(match.group(2))
            price_high = float(match.group(3))
            
            # Append a new row matching our exact PySpark schema from Step 3
            parsed_data.append((
                "Uvalde Auction",  # Auction_Location
                report_date,       # Sale_Date
                animal_class,      # Animal_Class
                "N/A",             # Weight_Range_lbs (Uvalde doesn't list weights for goats)
                price_low,         # Price_Low
                price_high,        # Price_High
                "cwt"              # Price_Unit (per hundredweight)
            ))

# 4. Convert the parsed list of rows into our official PySpark DataFrame
# (This uses the 'auction_schema' we defined in Cell 3)
final_df = spark.createDataFrame(parsed_data, schema=auction_schema)

# 5. Display the structured table!
display(final_df)